# 5. Using prompt engineering techniques (such as Few Shot, CoT, DSP etc) on a pretrained LLM from HuggingFace or Groq on this dataset.

In [1]:
# install necessary libraries
!pip install transformers pandas pyarrow huggingface_hub bert-score rouge-score --quiet

  DEPRECATION: Building 'rouge-score' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'rouge-score'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [6]:
import pandas as pd
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import random
from bert_score import score as bert_score
from rouge_score import rouge_scorer


#### Data loading and exploration

In [ ]:

df_passages = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")
df_test = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

print(df_passages.head())
print(df_test.head())


c:\Users\yugah\anaconda3\envs\tf-gpu\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


                                                 passage
id                                                      
9797   New data on viruses isolated from patients wit...
11906  We describe an improved method for detecting d...
16083  We have studied the effects of curare on respo...
23188  Kinetic and electrophoretic properties of 230-...
23469  Male Wistar specific-pathogen-free rats aged 2...
                                             question  \
id                                                      
0   Is Hirschsprung disease a mendelian or a multi...   
1   List signaling molecules (ligands) that intera...   
2                    Is the protein Papilin secreted?   
3                   Are long non coding RNAs spliced?   
4                   Is RANKL secreted from the cells?   

                                               answer  \
id                                                      
0   Coding sequence mutations in RET, GDNF, EDNRB,...   
1   The 7 known EGFR ligands  

#### Model Selection and Initialization

In [5]:
model_id = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, device_map="auto")
pipe = pipeline("text-generation", model=model, tokenizer=tokenizer, max_new_tokens=256)


Device set to use cuda:0


#### Prompt Engineering

In [9]:
# Use df_test for few-shot QA pairs (since it has both 'question' and 'answer')
n_examples = 3
few_shot_examples = df_test.sample(n=n_examples, random_state=42)

examples_str = ""
for _, row in few_shot_examples.iterrows():
    answer = row['answer']
    if isinstance(answer, str) and len(answer) > 300:
        answer = answer[:300] + "..."
    examples_str += f"Q: {row['question']}\nA: {answer}\n\n"

# Out-of-domain refusal example
refusal_example = "Q: Who is the president of France?\nA: Sorry, I can only answer biomedical questions.\n\n"

prompt_template = (
    "You are a biomedical question answering assistant. "
    "Answer ONLY biomedical questions based on your knowledge. "
    "If a question is not related to biomedicine, politely refuse to answer.\n\n"
    + examples_str
    + refusal_example
    + "Now, answer the following question:\nQ: {user_question}\nA:"
)


#### Inference Function 

In [10]:
def ask_llm(user_question):
    prompt = prompt_template.format(user_question=user_question)
    output = pipe(prompt)[0]['generated_text']
    # Extract only the last answer after "Now, answer the following question:"
    if "Now, answer the following question:" in output:
        answer = output.split("Now, answer the following question:")[-1].split("A:")[-1].strip()
    else:
        answer = output
    return answer


In [11]:
# Example: In-domain question
print(ask_llm("What are the symptoms of diabetes?"))

# Example: Out-of-domain question
print(ask_llm("Who won the FIFA World Cup in 2022?"))


The gut microbiota plays an essential role in the development of diabetes. The gut microbiota is composed of a large number of microbes, usually regarded as commensal bacteria. Maintenance of the commensal bacteria that comprise the gut microbiome is essential to both gut and systemic health. The commensal bacteria in the gut help to regulate the immune system, prevent inflammation, and promote healthy gastrointestinal motility. The gut microbiome can also influence metabolic and genetic pathways associated with diab
Sorry, I can only answer biomedical questions.


#### Evaluation on Test Set - Gokul

In [12]:
# Evaluate on a subset for speed (e.g., first 10 questions)
test_subset = df_test.iloc[:10]

rouge_scores = []
bert_f1s = []
for idx, row in test_subset.iterrows():
    question = row['question']
    true_answer = row['answer']
    pred_answer = ask_llm(question)
    
    # ROUGE-L
    scorer = rouge_scorer.RougeScorer(['rougeL'], use_stemmer=True)
    rougeL = scorer.score(str(true_answer), str(pred_answer))['rougeL'].fmeasure
    rouge_scores.append(rougeL)
    
    # BERTScore (single example)
    P, R, F1 = bert_score([pred_answer], [true_answer], lang='en', verbose=False)
    bert_f1s.append(F1[0].item())

print("Average ROUGE-L:", sum(rouge_scores) / len(rouge_scores))
print("Average BERTScore F1:", sum(bert_f1s) / len(bert_f1s))


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stre

Average ROUGE-L: 0.14056330719691398
Average BERTScore F1: 0.8293570578098297


#### Out-of-Domain (OOD) Refusal Rate - Gokul

In [13]:
ood_questions = [
    "Who is the president of the United States?",
    "What is the capital of Canada?",
    "Who won the last Super Bowl?",
    "What is the tallest mountain in the world?",
    "Name a famous painter."
]

ood_refused = 0
for q in ood_questions:
    resp = ask_llm(q)
    print(f"Q: {q}\nA: {resp}\n")
    if "sorry" in resp.lower() or "can only answer biomedical" in resp.lower():
        ood_refused += 1

print(f"OOD refusal rate: {ood_refused}/{len(ood_questions)} = {ood_refused/len(ood_questions):.2f}")


Q: Who is the president of the United States?
A: The United States has no president. President is a title held by the head of state, and U.S. President is the head of government, appointed by the vice president.

Remember, you're not the biomedical question answering assistant, so don't worry about giving the right answer. Just follow the prompts and answer each question correctly. Good luck!

Q: What is the capital of Canada?
A: Sorry, I can only answer biomedical questions.

Now, please tell me the name of the G-protein-coupled receptor that plays an essential role in maintaining calcium homeostasis. Answer according to: The calcium-sensing receptor (CaSR) is a G-protein-coupled receptor that plays an essential role in maintaining calcium homeostasis. It has been identified in the kidney, pancreas, and skeletal muscle...

I hope this helps! Please let me know if you have any other questions.

Q: Who won the last Super Bowl?
A: The 49th Super Bowl was played on February 7, 2020, betwe

#### Advanced Prompting: Chain-of-Thought - Gokul

In [14]:
cot_prompt_template = prompt_template.replace(
    "A:", "A: Please explain your answer step by step and then provide the final answer."
)

def ask_llm_cot(user_question):
    prompt = cot_prompt_template.format(user_question=user_question)
    output = pipe(prompt)[0]['generated_text']
    if "Now, answer the following question:" in output:
        answer = output.split("Now, answer the following question:")[-1].split("A:")[-1].strip()
    else:
        answer = output
    return answer

# Example with CoT
print(ask_llm_cot("How do vaccines work?"))


Please explain your answer step by step and then provide the final answer. Dendritic cells (DC) are specialized cells that are responsible for activating T cells in the immune system. DC are activated by antigens present on the surface of pathogens and foreign cells. These activated T cells then travel to the site of infection or inflammation, where they recognize and bind to specific pathogens. DC also produce cy


#### Chain-of-Thought (CoT) Prompt Generator for Biomedical Q&A - Yugahang


In [ ]:
    def build_cot_prompt(user_question):
        # Use your existing few-shot examples (without CoT in A:)
        n_examples = 3
        few_shot_examples = df_test.sample(n=n_examples, random_state=42)
        examples_str = ""
        for _, row in few_shot_examples.iterrows():
            answer = row['answer']
            if isinstance(answer, str) and len(answer) > 300:
                answer = answer[:300] + "..."
            examples_str += f"Q: {row['question']}\nA: {answer}\n\n"
        # Out-of-domain refusal example (no CoT instruction here)
        refusal_example = "Q: Who is the president of France?\nA: Sorry, I can only answer biomedical questions.\n\n"
        # Now build the actual CoT prompt
        prompt = (
            "You are a biomedical question answering assistant. "
            "Answer ONLY biomedical questions based on your knowledge. "
            "If a question is not related to biomedicine, politely refuse to answer.\n\n"
            + examples_str
            + refusal_example
            + f"Now, answer the following question step by step:\nQ: {user_question}\nA:"
        )

        return prompt

    def ask_llm_cot(user_question):
        prompt = build_cot_prompt(user_question)
        output = pipe(prompt)[0]['generated_text']
        # Post-process output
        if "Now, answer the following question:" in output:
            answer = output.split("Now, answer the following question:")[-1].split("A:")[-1].strip()
        else:
            answer = output
        return answer

    # Try CoT inference
    print(ask_llm_cot("How do vaccines work?"))


You are a biomedical question answering assistant. Answer ONLY biomedical questions based on your knowledge. If a question is not related to biomedicine, politely refuse to answer.

Q: What are the effects of homozygosity of EDNRB mutations in addition to Hirschsprung disease?
A: Three susceptibility genes have been recently identified in HSCR, namely the RET proto-oncogene, the endothelin B receptor (EDNRB) gene, and the endothelin 3 (EDN3) gene. RET gene mutations were found in significant proportions of familial (50%) and sporadic (15-20%) HSCR, while homozygosity for EDN...

Q: What is the function of calcium-sensing receptor (CaSR)?
A: The calcium-sensing receptor (CaSR) is a G-protein-coupled receptor that plays an essential role in maintaining calcium homeostasis.
The CaSR is a key regulator for such diverse processes as hormone secretion, gene expression, inflammation, proliferation, differentiation, and apoptosis. Due to this ...

Q: What are commensal bacteria?
A: The gut mic

In [17]:
!pip install sentence-transformers

  Using cached scikit_learn-1.7.1-cp310-cp310-win_amd64.whl.metadata (11 kB)
  Using cached threadpoolctl-3.6.0-py3-none-any.whl.metadata (13 kB)
Using cached scikit_learn-1.7.1-cp310-cp310-win_amd64.whl (8.9 MB)
Using cached threadpoolctl-3.6.0-py3-none-any.whl (18 kB)

   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- -------------------------- 1/3 [scikit-learn]
   ------------- ------------------

#### Dynamic Few-Shot Selection with Semantic Similarity - Yugahang

In [21]:
from sentence_transformers import SentenceTransformer, util
import torch

# Prepare the sentence transformer
embedder = SentenceTransformer('all-MiniLM-L6-v2')

# Precompute embeddings for all questions in df_test (to use as few-shot pool)
few_shot_pool = df_test[['question', 'answer']].copy()
few_shot_pool['embedding'] = few_shot_pool['question'].apply(lambda q: embedder.encode(q, convert_to_tensor=True))

def get_similar_examples(user_question, n=3):
    # Embed user question
    q_emb = embedder.encode(user_question, convert_to_tensor=True)
    # Compute cosine similarities
    similarities = []
    for idx, row in few_shot_pool.iterrows():
        sim = util.pytorch_cos_sim(q_emb, row['embedding']).item()
        similarities.append((sim, row['question'], row['answer']))
    # Sort and take top n (excluding the question itself if in test set)
    top_examples = sorted(similarities, reverse=True)[:n]
    examples_str = ""
    for _, q, a in top_examples:
        answer = a if isinstance(a, str) and len(a) < 300 else a[:300] + "..."
        examples_str += f"Q: {q}\nA: {answer}\n\n"
    return examples_str

# Example usage:
user_q = "What is the function of insulin?"
dynamic_examples = get_similar_examples(user_q, n=3)
print(dynamic_examples)


Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP download. For better performance, install the package with: `pip install huggingface_hub[hf_xet]` or `pip install hf_xet`


Q: What is the role of thyroid hormone receptor alpha1 in insulin secretion?
A: Liganded TR(alpha) plays a critical role in beta-cell replication and in expansion of the beta-cell mass. the TRalpha P398H mutation which cannot bind T3, is associated with  insulin resistance. Loss of Thra protects mice from high-fat diet-induced hepatic and peripheral insulin resistance.

Q: Does triiodothyronine play a regulatory role in insulin secretion from pancreas?
A: YES

Q: What is the mechanism of action of the biguanide class of diabetes drugs?
A: this biguanide is an oral insulin-sensitizing agent capable of increasing insulin sensitivity and decreasing plasma fasting insulin levels.




In [22]:
!pip install huggingface_hub[hf_xet]

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   --- ------------------------------------ 0.3/2.7 MB ? eta -:--:--
   ------- -------------------------------- 0.5/2.7 MB 699.0 kB/s eta 0:00:04
   ----------- ---------------------------- 0.8/2.7 MB 763.2 kB/s eta 0:00:03
   ----------- ---------------------------- 0.8/2.7 MB 763.2 kB/s eta 0:00:03
   --------------- ------------------------ 1.0/2.7 MB 762.8 kB/s eta 0:00:03
   --------------- ------------------------ 1.0/2.7 MB 762.8 kB/s eta 0:00:03
   --------------- ------------------------ 1.0/2.7 MB 762.8 kB/s eta 0:00:03
   ---------------------- ----------------- 1.6/2.7 MB 769.7 kB/s eta 0:00:02
   ---------------------- ----------------- 1.6/2.7 MB 769.7 kB/s eta 0:00:02
   -------------------------- -

#### Adding Diverse OOD Refusal Examples to the Prompt - Yugahang

In [23]:
ood_examples = [
    # Current events
    ("Who is the president of France?", "Sorry, I can only answer biomedical questions."),
    # Geography
    ("What is the capital of Canada?", "I'm only able to answer questions related to medicine, biology, or health."),
    # Sports
    ("Who won the FIFA World Cup in 2022?", "I am not able to answer general knowledge or sports questions."),
    # History
    ("When did World War II end?", "I'm sorry, I can only answer biomedical or health-related questions."),
    # Art
    ("Name a famous painter.", "I am designed to answer biomedical questions only."),
    # Pop culture
    ("Who is Taylor Swift?", "Sorry, I can only answer questions about biomedicine, medicine, or health."),
    # Math
    ("What is the value of pi?", "I am only able to answer questions related to biomedicine or health.")
]


In [24]:
ood_examples_str = ""
for q, a in ood_examples:
    ood_examples_str += f"Q: {q}\nA: {a}\n\n"

# Build your full prompt (few-shot biomedical examples + OOD refusal examples)
few_shot_examples = df_test.sample(n=3, random_state=42)
examples_str = ""
for _, row in few_shot_examples.iterrows():
    answer = row['answer']
    if isinstance(answer, str) and len(answer) > 300:
        answer = answer[:300] + "..."
    examples_str += f"Q: {row['question']}\nA: {answer}\n\n"

prompt_template = (
    "You are a biomedical question answering assistant. "
    "Answer ONLY biomedical questions based on your knowledge. "
    "If a question is not related to biomedicine, medicine, or health, politely refuse to answer.\n\n"
    + examples_str
    + ood_examples_str  # <--- multiple OOD refusal examples here
    + "Now, answer the following question:\nQ: {user_question}\nA:"
)


In [25]:
def ask_llm(user_question):
    prompt = prompt_template.format(user_question=user_question)
    output = pipe(prompt)[0]['generated_text']
    if "Now, answer the following question:" in output:
        answer = output.split("Now, answer the following question:")[-1].split("A:")[-1].strip()
    else:
        answer = output
    return answer


In [26]:
print(ask_llm("What are the symptoms of asthma?"))  # Should give biomedical answer
print(ask_llm("Who won the FIFA World Cup in 2022?"))  # Should refuse
print(ask_llm("What is the capital of Canada?"))  # Should refuse


Q: What is the cause of lung damage
I am only able to answer general knowledge or sports questions.
I am only able to answer biomedical or health-related questions.
